# 03 — Combine three tables → one dataset + missing-value clean

**Goal of this notebook**
1. Put `linkedin_job_postings` + `job_skills` + `job_summary` into **one** table
2. Use `job_link` as the join / index key
3. Do an overall missing-value clean
4. Report **how many rows × columns** we end up with

**Not in this notebook**
- No EDA yet
- No splitting of the long `job_skills` / `job_summary` text into many columns yet  
  (those two fields are intentionally left as long text for a later parsing step)

| Source | What it adds on each `job_link` |
|---|---|
| postings | many structured columns (title, company, location, level, …) |
| skills | long comma-separated skill text |
| summary | long job-description text |


## 0. Setup


In [ ]:
from pathlib import Path
import warnings

import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

# Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/Arusto_LinkedIn')

DATA_DIR = Path(".")
OUT_DIR = DATA_DIR / "cleaned"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POSTINGS_PATH = DATA_DIR / "linkedin_job_postings.csv"
SKILLS_PATH   = DATA_DIR / "job_skills.csv"
SUMMARY_PATH  = DATA_DIR / "job_summary.csv"

# summary is ~4.8GB — chunked read keeps RAM calmer
SUMMARY_CHUNKSIZE = 50_000

for p in [POSTINGS_PATH, SKILLS_PATH, SUMMARY_PATH]:
    assert p.exists(), f"missing: {p}"
    print(f"OK  {p.name:30} {p.stat().st_size/1e6:7.0f} MB")


In [ ]:
def tf_to_bool(series):
    return series.astype("string").str.lower().map({"t": True, "f": False})


def strip_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip()
    return df.replace({"": pd.NA})


def save_df(df, name):
    # index (job_link) should be kept in the file
    pq = OUT_DIR / f"{name}.parquet"
    gz = OUT_DIR / f"{name}.csv.gz"
    try:
        df.to_parquet(pq)
        print(f"saved {pq.name}  ({df.shape[0]:,} × {df.shape[1]}, {pq.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"parquet failed ({e}); writing csv.gz")
        df.to_csv(gz, compression="gzip")
        print(f"saved {gz.name}  ({df.shape[0]:,} × {df.shape[1]}, {gz.stat().st_size/1e6:.1f} MB)")


## 1. Load each table (still separate)

Light per-table tidy only — enough so the key is clean before we stack them.


In [ ]:
print("loading postings...")
postings = pd.read_csv(POSTINGS_PATH)
print("raw postings:", postings.shape)

postings = strip_cols(postings, [
    "job_link", "job_title", "company", "job_location",
    "search_city", "search_country", "search_position",
    "job_level", "job_type",
])

for c in ["got_summary", "got_ner", "is_being_worked"]:
    postings[c] = tf_to_bool(postings[c])

postings["first_seen"] = pd.to_datetime(postings["first_seen"], errors="coerce")
postings["last_processed_time"] = pd.to_datetime(
    postings["last_processed_time"], errors="coerce", utc=True
)
postings["job_level"] = postings["job_level"].str.title()
postings["job_type"] = postings["job_type"].str.title()

# key must exist; one row per job_link
postings = postings.dropna(subset=["job_link"]).drop_duplicates("job_link", keep="first")
print("tidy postings:", postings.shape)
postings.head(2)


In [ ]:
print("loading skills...")
skills = pd.read_csv(SKILLS_PATH)
print("raw skills:", skills.shape)

skills = strip_cols(skills, ["job_link", "job_skills"])
skills = skills.dropna(subset=["job_link"]).drop_duplicates("job_link", keep="first")

# keep the long text as-is for now (split/classify later)
# just note whether the cell is empty
skills["skills_is_missing"] = skills["job_skills"].isna()

print("tidy skills:", skills.shape)
print("example skills text (first 200 chars):")
print(str(skills["job_skills"].dropna().iloc[0])[:200])
skills.head(2)


In [ ]:
print("loading summary in chunks (this takes a bit)...")

summary_parts = []
n_raw = 0

for i, chunk in enumerate(pd.read_csv(SUMMARY_PATH, chunksize=SUMMARY_CHUNKSIZE)):
    n_raw += len(chunk)
    chunk = strip_cols(chunk, ["job_link", "job_summary"])
    chunk = chunk.dropna(subset=["job_link"]).drop_duplicates("job_link", keep="first")
    summary_parts.append(chunk)
    if (i + 1) % 5 == 0:
        print(f"  ... chunks={i+1}, raw rows so far={n_raw:,}", flush=True)

summary = pd.concat(summary_parts, ignore_index=True)
# in case the same link appeared across chunk boundaries
summary = summary.drop_duplicates("job_link", keep="first")
summary["summary_is_missing"] = summary["job_summary"].isna()

print("raw summary rows read:", f"{n_raw:,}")
print("tidy summary:", summary.shape)
print("example summary text (first 200 chars):")
print(str(summary["job_summary"].dropna().iloc[0])[:200])
summary.head(2)


## 2. Combine on `job_link` → one dataset

- Start from **postings** (the spine)
- Left-join **skills**, then **summary**
- Set `job_link` as the DataFrame **index**


In [ ]:
# avoid bringing duplicate key columns; rename long-text fields clearly if needed
skills_j = skills[["job_link", "job_skills", "skills_is_missing"]].copy()
summary_j = summary[["job_link", "job_summary", "summary_is_missing"]].copy()

combined = (
    postings
    .merge(skills_j, on="job_link", how="left")
    .merge(summary_j, on="job_link", how="left")
)

# job_link becomes the index key
combined = combined.set_index("job_link", drop=True)
combined.index.name = "job_link"

print("combined shape (rows × cols):", combined.shape)
print("index unique?", combined.index.is_unique)
combined.head(2)


## 3. Overall missing-value clean

What we do here (conservative — we still need the long text later):

1. Report missingness **before** cleaning  
2. Drop rows with no usable identity (`job_title` empty)  
3. Treat empty strings in long text as missing  
4. Add simple coverage flags (`has_skills`, `has_summary`)  
5. Do **not** drop rows just because skills/summary text is missing — that would throw away a lot of postings metadata  
6. Do **not** explode / tokenize skills or summary yet


In [ ]:
print("MISSINGNESS BEFORE CLEAN")
print("=" * 40)
miss = combined.isna().sum().sort_values(ascending=False)
miss_pct = (miss / len(combined) * 100).round(2)
before_tbl = pd.DataFrame({"missing": miss, "pct": miss_pct})
display(before_tbl)


In [ ]:
n_before = len(combined)

# empty long text → NA (so missing counts are honest)
for col in ["job_skills", "job_summary", "job_title", "company", "job_location"]:
    if col in combined.columns:
        combined[col] = combined[col].astype("string").str.strip()
        combined.loc[combined[col] == "", col] = pd.NA

# rows without a title are not useful for "what courses to build"
combined = combined.dropna(subset=["job_title"])

# coverage flags for later filtering (analysis still deferred)
combined["has_skills"] = combined["job_skills"].notna()
combined["has_summary"] = combined["job_summary"].notna()

# if the helper cols from the side tables are NA after left join, fill flags
if "skills_is_missing" in combined.columns:
    combined["skills_is_missing"] = combined["skills_is_missing"].fillna(True).astype(bool)
if "summary_is_missing" in combined.columns:
    combined["summary_is_missing"] = combined["summary_is_missing"].fillna(True).astype(bool)

n_after = len(combined)
print(f"rows: {n_before:,} → {n_after:,}  (dropped {n_before - n_after:,} without job_title)")
print()
print("coverage")
print("  has_skills :", f"{combined['has_skills'].mean():.1%}", f"({combined['has_skills'].sum():,})")
print("  has_summary:", f"{combined['has_summary'].mean():.1%}", f"({combined['has_summary'].sum():,})")
print("  both       :", f"{(combined['has_skills'] & combined['has_summary']).mean():.1%}")


In [ ]:
print("MISSINGNESS AFTER CLEAN")
print("=" * 40)
miss = combined.isna().sum().sort_values(ascending=False)
miss_pct = (miss / len(combined) * 100).round(2)
after_tbl = pd.DataFrame({"missing": miss, "pct": miss_pct})
display(after_tbl)

print()
print("columns:")
for c in combined.columns:
    print(f"  - {c}")


## 4. Final size check (the number you asked for)


In [ ]:
n_rows, n_cols = combined.shape

print("FINAL COMBINED DATASET")
print("=" * 40)
print(f"rows   : {n_rows:,}")
print(f"columns: {n_cols}")
print(f"index  : {combined.index.name}  (unique={combined.index.is_unique})")
print()
print("column list:")
print(list(combined.columns))
print()
print("dtypes:")
print(combined.dtypes)


In [ ]:
# peek at the two long-text fields we'll parse later — still single columns for now
print("job_skills sample:")
print(str(combined.loc[combined["has_skills"], "job_skills"].iloc[0])[:300])
print()
print("job_summary sample:")
print(str(combined.loc[combined["has_summary"], "job_summary"].iloc[0])[:300])


## 5. Save


In [ ]:
save_df(combined, "jobs_combined")

# smaller US slice for later work (still no EDA here)
if "search_country" in combined.columns:
    us = combined.loc[combined["search_country"] == "United States"].copy()
    save_df(us, "jobs_combined_us")
    print(f"US slice: {us.shape[0]:,} × {us.shape[1]}")

print()
print("Next (later notebooks): split/classify job_skills & job_summary text → then EDA.")
